In [ ]:
from pathlib import Path
from clinical_notes_extraction.config import PROJECT_ROOT
import os

import numpy as np
import pandas as pd

In [ ]:
# Constants local to this notebook
DATA_PATH = Path(f"{PROJECT_ROOT}/scripts/3_information_extraction/3_2_medications_on_admission/data")

EMBEDDINGS_FILE = Path(f"{DATA_PATH}/embeddings.npy")        # population embeddings, row-aligned with POPULATION_FILE
POPULATION_FILE = Path(f"{DATA_PATH}/final_dataset.parquet")   # note_id, cluster, text (output of the clustering notebook)
SAMPLE_DIR = Path(f"{DATA_PATH}/sample.parquet")                     # to exclude sample notes from medoid candidates

# NOTE: the medoid search should run in the SAME representation used by KMeans.
# If clustering was done on the PCA-reduced, L2-renormalised matrix, point
# EMBEDDINGS_FILE at that matrix (or re-apply the saved PCA transform here).

In [ ]:
embeddings = np.load(EMBEDDINGS_FILE)
population = pd.read_parquet(POPULATION_FILE)

assert len(embeddings) == len(population), (
    "Embeddings and population dataframe must be row-aligned"
)

# The population parquet kept stale index labels from an earlier filtering step.
# Reset it so index labels match row positions in the embeddings array.
population = population.reset_index(drop=True)

# All 32 sample notes (dev + prod) are excluded as medoid candidates,
# so no note ever appears both as a dynamic prompt example and as an evaluation note.
sample_ids = pd.read_parquet(SAMPLE_DIR)["note_id"]

# Filter dataframe and embeddings with the same boolean mask so both stay row-aligned.
mask = population["note_id"].isin(sample_ids)
cleaned_population = population[mask].reset_index(drop=True)
cleaned_embeddings = embeddings[mask.to_numpy()]

assert len(cleaned_population) == len(cleaned_embeddings)

print(
    f"{len(population)} population notes | "
    f"{len(sample_ids)} sample notes excluded | "
    f"{len(cleaned_population)} cleaned population notes that will be used to get the medoids"
)

In [ ]:
cleaned_population

In [ ]:
cleaned_embeddings

In [ ]:
sample = cleaned_population.copy()

sample

In [ ]:
sample_embeddings_dict = pd.Series(
    cleaned_population.index, index=cleaned_population["note_id"]
).to_dict()

sample["embedding"] = sample["note_id"].map(
    lambda nid: cleaned_embeddings[sample_embeddings_dict[nid]] if nid in sample_embeddings_dict else None
)

df_sample_embeddings = sample.copy()
df_sample_embeddings



In [ ]:
DEV_SAMPLE = Path(f"{DATA_PATH}/dev_sample.parquet")   # note_id, cluster, text (output of the clustering notebook)

dev_sample = pd.read_parquet(DEV_SAMPLE)


In [ ]:
dev_sample_embeddings = dev_sample.merge(
    df_sample_embeddings[["note_id", "embedding"]],
    on="note_id",
    how="left",
)

dev_sample_embeddings

In [ ]:
dev_sample_embeddings.to_parquet(f"{DATA_PATH}/dev_sample_final.parquet")

----

In [ ]:
PROD_SAMPLE = Path(f"{DATA_PATH}/prod_sample.parquet")

prod_sample = pd.read_parquet(PROD_SAMPLE)


In [ ]:
prod_sample_embeddings = prod_sample.merge(
    df_sample_embeddings[["note_id", "embedding"]],
    on="note_id",
    how="left",
)

prod_sample_embeddings

In [ ]:
prod_sample_embeddings.to_parquet(f"{DATA_PATH}/prod_sample_final.parquet")